In [19]:
from pyspark import SparkContext
from pyspark.sql import SparkSession

spark = SparkSession\
        .builder\
        .master('local[*]')\
        .getOrCreate()
sc = spark.sparkContext

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
%cd /content/drive/My Drive/Colab Notebooks/Big Data Notebooks/HandsOn_10.1_Classification
!ls

/content/drive/My Drive/Colab Notebooks/Big Data Notebooks/HandsOn_10.1_Classification
10.2.Spark-ClassificationDajiaForbes.ipynb  daily_weather.csv
10.Spark-ClassificationDajiaForbes.ipynb


In [22]:
from pyspark.sql import SQLContext
from pyspark.sql import DataFrameNaFunctions
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.feature import Binarizer
from pyspark.ml.feature import VectorAssembler,StringIndexer,VectorIndexer

In [23]:
SQLContext = SQLContext(sc)
df = SQLContext.read.load('/content/drive/My Drive/Colab Notebooks/Big Data Notebooks/HandsOn_10.1_Classification/daily_weather.csv',
                          format = 'com.databricks.spark.csv',
                          header = 'true',inferSchema = 'true')
df.columns

['number',
 'air_pressure_9am',
 'air_temp_9am',
 'avg_wind_direction_9am',
 'avg_wind_speed_9am',
 'max_wind_direction_9am',
 'max_wind_speed_9am',
 'rain_accumulation_9am',
 'rain_duration_9am',
 'relative_humidity_9am',
 'relative_humidity_3pm']

In [24]:
featureColumns = ['air_pressure_9am','air_temp_9am','avg_wind_direction_9am', 'avg_wind_speed_9am','max_wind_direction_9am',
                   'max_wind_speed_9am','rain_accumulation_9am','rain_duration_9am']

In [25]:
df = df.drop('number')

In [26]:
df = df.na.drop()

In [27]:
df.count() , len(df.columns)

(1064, 10)

In [28]:
Binarizer = Binarizer(threshold = 24.99999,inputCol ="relative_humidity_3pm",outputCol = "label")
BinarizerDF = Binarizer.transform(df)

In [29]:
BinarizerDF.select("relative_humidity_3pm","label").show(4)

+---------------------+-----+
|relative_humidity_3pm|label|
+---------------------+-----+
|   36.160000000000494|  1.0|
|     19.4265967985621|  0.0|
|   14.460000000000045|  0.0|
|   12.742547353761848|  0.0|
+---------------------+-----+
only showing top 4 rows



In [30]:
assembler = VectorAssembler(inputCols=featureColumns , outputCol="features")
assembled = assembler.transform(BinarizerDF)

In [31]:
(trainingData,testData) = assembled.randomSplit([0.8,0.2],seed = 13234)

In [32]:
trainingData.count() ,testData.count()

(846, 218)

In [33]:
dt = DecisionTreeClassifier(labelCol="label",featuresCol="features",maxDepth=5,minInstancesPerNode=20,impurity="gini")

In [34]:
pipeline = Pipeline(stages=[dt])
model = pipeline.fit(trainingData)
predictions = model.transform(testData)

In [35]:
predictions.select("prediction","label").show(10)

+----------+-----+
|prediction|label|
+----------+-----+
|       1.0|  1.0|
|       1.0|  1.0|
|       0.0|  1.0|
|       1.0|  1.0|
|       1.0|  1.0|
|       1.0|  1.0|
|       1.0|  1.0|
|       1.0|  1.0|
|       0.0|  0.0|
|       1.0|  1.0|
+----------+-----+
only showing top 10 rows



In [36]:
predictions.select("prediction","label").write.save(path = "/content/drive/My Drive/Colab Notebooks/Big Data Notebooks/HandsOn_10.1_Classification/predictions.csv",
                                                    format = "com.databricks.spark.csv",
                                                    header='true')